In [2]:
!pip install -q timm



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# HAM10000 streaming training (NO full dataset download)
# Uses torchvision ResNet18 (no timm)
# Stable streaming: num_workers=0, robust skipping of bad samples

import io
import itertools
import random
from PIL import Image

import requests
import torch
from torch import nn
from torch.utils.data import IterableDataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from datasets import load_dataset

# ------------------ CONFIG ------------------
DATASET_ID = "kuchikihater/HAM10000"   # HF mirror (streamable)
SPLIT = "train"                        # most mirrors provide a train split
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LR = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# For streaming, keep this 0 to avoid worker crashes from network/PIL issues
NUM_WORKERS = 0

# Limit samples per epoch so each epoch finishes in predictable time
TRAIN_SAMPLES_PER_EPOCH = 6000
VAL_SAMPLES_PER_EPOCH   = 1200

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
# -------------------------------------------

# HAM10000 class mapping (7 classes)
LABEL2ID = {"nv": 0, "mel": 1, "bkl": 2, "bcc": 3, "akiec": 4, "vasc": 5, "df": 6}
NUM_CLASSES = len(LABEL2ID)

# Image transforms
train_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(0.1, 0.1, 0.1, 0.05),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class HAMStream(IterableDataset):
    """
    Streams HAM10000 from Hugging Face (no full download).
    Robustly skips any problematic items (bad label/image/url).
    """
    def __init__(self, split="train", transforms=None, max_samples=None):
        super().__init__()
        self.transforms = transforms
        self.max_samples = max_samples

        # streaming=True ensures no full dataset download
        self.ds = load_dataset(DATASET_ID, split=split, streaming=True)

        # common label keys across mirrors
        self.label_keys = ["dx", "label", "class", "target"]

        # reuse HTTP connection
        self.session = requests.Session()

    def _get_label(self, ex):
        for k in self.label_keys:
            if k in ex:
                v = ex[k]
                if isinstance(v, int):
                    return v
                if isinstance(v, str):
                    return LABEL2ID.get(v, None)
                if isinstance(v, list) and v:
                    if isinstance(v[0], str):
                        return LABEL2ID.get(v[0], None)
        # fallback metadata
        md = ex.get("meta", {}) or ex.get("metadata", {})
        if isinstance(md, dict):
            dx = md.get("dx") or md.get("diagnosis")
            if isinstance(dx, str):
                return LABEL2ID.get(dx, None)
        return None

    def _load_image_from_url(self, url):
        # Download and decode image safely
        r = self.session.get(url, timeout=10)
        r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")

    def _get_image(self, ex):
        # Try common image fields
        img_field = ex.get("image") or ex.get("img")

        # Sometimes it's already a PIL image
        if isinstance(img_field, Image.Image):
            return img_field.convert("RGB")

        # Sometimes it's a dict containing url
        if isinstance(img_field, dict) and "url" in img_field and isinstance(img_field["url"], str):
            return self._load_image_from_url(img_field["url"])

        # Sometimes it's raw bytes
        if isinstance(img_field, (bytes, bytearray)):
            return Image.open(io.BytesIO(img_field)).convert("RGB")

        # Some mirrors provide image_url / url
        url = ex.get("image_url") or ex.get("url")
        if isinstance(url, str):
            return self._load_image_from_url(url)

        return None

    def __iter__(self):
        # NOTE: we keep NUM_WORKERS=0, so worker sharding isn't required.
        # If you later want NUM_WORKERS>0, add itertools.islice sharding here.
        count = 0
        for ex in self.ds:
            if self.max_samples is not None and count >= self.max_samples:
                break
            try:
                label = self._get_label(ex)
                if label is None:
                    continue

                img = self._get_image(ex)
                if img is None:
                    continue

                if self.transforms:
                    img = self.transforms(img)

                yield img, int(label)
                count += 1

            except Exception:
                # Robust skip on any failure (network, PIL decode, etc.)
                continue


def build_model():
    # Pretrained ResNet18, replace final layer for 7-class classification
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


def train_one_epoch(model, loader, opt, criterion):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += x.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        logits = model(x)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += x.size(0)
    return correct / max(total, 1)


def main():
    print("Device:", DEVICE)

    train_ds = HAMStream(split=SPLIT, transforms=train_tfms, max_samples=TRAIN_SAMPLES_PER_EPOCH)
    val_ds   = HAMStream(split=SPLIT, transforms=val_tfms,   max_samples=VAL_SAMPLES_PER_EPOCH)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

    model = build_model().to(DEVICE)

    # Basic loss; for better performance later, we can add class weights or focal loss
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, opt, criterion)
        val_acc = evaluate(model, val_loader)
        print(f"Epoch {epoch}/{EPOCHS} | loss={train_loss:.4f} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

    torch.save(model.state_dict(), "ham10000_resnet18_stream.pth")
    print("Saved: ham10000_resnet18_stream.pth")


if __name__ == "__main__":
    main()


Device: cpu
Epoch 1/5 | loss=1.0486 | train_acc=0.6875 | val_acc=0.0008
Epoch 2/5 | loss=1.1366 | train_acc=0.6567 | val_acc=0.0008


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: a9aa7f36-8403-43ae-8082-f10e59633707)')' thrown while requesting GET https://huggingface.co/datasets/kuchikihater/HAM10000/resolve/24ca17aae90fbe18ac412f5de5cf95965584af9e/data/train-00002-of-00006-39dedb0b31996810.parquet
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: edc50ab7-c4a7-4ffd-bfd5-35e72a08bf07)')' thrown while requesting GET https://huggingface.co/datasets/kuchikihater/HAM10000/resolve/24ca17aae90fbe18ac412f5de5cf95965584af9e/data/train-00002-of-00006-39dedb0b31996810.parquet
Retrying in 2s [Retry 2/5].
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 89c59ec3-a5bc-46f8-91f4-0c9fc2d02821)')' thrown while requesting GET https://huggingface.co/datasets/kuchikihater/HAM10000/resolv

Epoch 3/5 | loss=1.1711 | train_acc=0.6477 | val_acc=0.0008


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: bcc29221-5bc2-4209-a381-492b2f14210b)')' thrown while requesting GET https://huggingface.co/datasets/kuchikihater/HAM10000/resolve/24ca17aae90fbe18ac412f5de5cf95965584af9e/data/train-00001-of-00006-d0e18093f7a3c694.parquet
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: dcf39a6a-560b-4925-a60d-b90a44c30bae)')' thrown while requesting GET https://huggingface.co/datasets/kuchikihater/HAM10000/resolve/24ca17aae90fbe18ac412f5de5cf95965584af9e/data/train-00001-of-00006-d0e18093f7a3c694.parquet
Retrying in 2s [Retry 2/5].
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 0c85ed45-6fef-492b-a157-5ad6da7e5b8e)')' thrown while requesting GET https://huggingface.co/datasets/kuchikihater/HAM10000/resolv

Epoch 4/5 | loss=1.2549 | train_acc=0.6190 | val_acc=0.0008
Epoch 5/5 | loss=1.3149 | train_acc=0.5672 | val_acc=0.0008
Saved: ham10000_resnet18_stream.pth


In [5]:
# ============================================================
# HAM10000 Streaming Skin Lesion Classifier (ResNet18, PyTorch)
# - Streams HAM10000 from Hugging Face (no full dataset download)
# - Trains a 7-class classifier
# - Saves weights to: ham10000_resnet18_stream.pth
# - Includes inference code to predict on your own image
# ============================================================

import io
import random
from PIL import Image

import requests
import torch
from torch import nn
from torch.utils.data import IterableDataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from datasets import load_dataset

# ------------------ CONFIG ------------------
DATASET_ID = "kuchikihater/HAM10000"   # HF mirror (streamable)
SPLIT = "train"                        # most mirrors provide a train split

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LR = 2e-4

# Keep 0 for stability with streaming/network image fetching
NUM_WORKERS = 0

# Limit samples per epoch (keeps epochs predictable in streaming mode)
TRAIN_SAMPLES_PER_EPOCH = 6000
VAL_SAMPLES_PER_EPOCH   = 1200

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# -------------------------------------------

random.seed(SEED)
torch.manual_seed(SEED)

# HAM10000 class mapping (7 classes)
LABEL2ID = {"nv": 0, "mel": 1, "bkl": 2, "bcc": 3, "akiec": 4, "vasc": 5, "df": 6}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = len(LABEL2ID)

# Image transforms (match ImageNet normalization because we use pretrained ResNet)
train_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(0.1, 0.1, 0.1, 0.05),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class HAMStream(IterableDataset):
    """
    Streams HAM10000 from Hugging Face (no full download).
    Robustly skips problematic examples (bad label/image/url).
    """
    def __init__(self, split="train", transforms=None, max_samples=None):
        super().__init__()
        self.transforms = transforms
        self.max_samples = max_samples

        # streaming=True => fetch samples on-the-fly
        self.ds = load_dataset(DATASET_ID, split=split, streaming=True)

        # label keys vary by mirror; try common ones
        self.label_keys = ["dx", "label", "class", "target"]

        # HTTP session reuse for speed
        self.session = requests.Session()

    def _get_label(self, ex):
        # Try a few common label keys
        for k in self.label_keys:
            if k in ex:
                v = ex[k]
                if isinstance(v, int):
                    return v
                if isinstance(v, str):
                    return LABEL2ID.get(v, None)
                if isinstance(v, list) and v:
                    if isinstance(v[0], str):
                        return LABEL2ID.get(v[0], None)

        # fallback metadata
        md = ex.get("meta", {}) or ex.get("metadata", {})
        if isinstance(md, dict):
            dx = md.get("dx") or md.get("diagnosis")
            if isinstance(dx, str):
                return LABEL2ID.get(dx, None)
        return None

    def _load_image_from_url(self, url):
        r = self.session.get(url, timeout=10)
        r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")

    def _get_image(self, ex):
        img_field = ex.get("image") or ex.get("img")

        # Already a PIL image?
        if isinstance(img_field, Image.Image):
            return img_field.convert("RGB")

        # Dict with url?
        if isinstance(img_field, dict) and "url" in img_field and isinstance(img_field["url"], str):
            return self._load_image_from_url(img_field["url"])

        # Raw bytes?
        if isinstance(img_field, (bytes, bytearray)):
            return Image.open(io.BytesIO(img_field)).convert("RGB")

        # Some mirrors use image_url/url fields
        url = ex.get("image_url") or ex.get("url")
        if isinstance(url, str):
            return self._load_image_from_url(url)

        return None

    def __iter__(self):
        count = 0
        for ex in self.ds:
            if self.max_samples is not None and count >= self.max_samples:
                break
            try:
                label = self._get_label(ex)
                if label is None:
                    continue

                img = self._get_image(ex)
                if img is None:
                    continue

                if self.transforms:
                    img = self.transforms(img)

                yield img, int(label)
                count += 1

            except Exception:
                # Skip anything that fails (timeouts, corrupted images, etc.)
                continue


def build_model():
    # Pretrained ResNet18; replace classification head for 7 classes
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


def train_one_epoch(model, loader, opt, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += x.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        logits = model(x)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += x.size(0)
    return correct / max(total, 1)


def train_and_save():
    print("Device:", DEVICE)

    train_ds = HAMStream(split=SPLIT, transforms=train_tfms, max_samples=TRAIN_SAMPLES_PER_EPOCH)
    val_ds   = HAMStream(split=SPLIT, transforms=val_tfms,   max_samples=VAL_SAMPLES_PER_EPOCH)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

    model = build_model().to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, opt, criterion)
        val_acc = evaluate(model, val_loader)
        print(f"Epoch {epoch}/{EPOCHS} | loss={train_loss:.4f} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

    torch.save(model.state_dict(), "ham10000_resnet18_stream.pth")
    print("Saved: ham10000_resnet18_stream.pth")


# ------------------ INFERENCE ------------------
def load_trained_model(weights_path="ham10000_resnet18_stream.pth"):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    state = torch.load(weights_path, map_location=DEVICE)
    model.load_state_dict(state)
    model = model.to(DEVICE)
    model.eval()
    return model

@torch.no_grad()
def predict_image(model, image_path):
    """
    Predict lesion class for a local image file path.
    Returns: (pred_label, confidence, probs_tensor)
    """
    img = Image.open(image_path).convert("RGB")
    x = val_tfms(img).unsqueeze(0).to(DEVICE)  # [1,3,224,224]

    logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]
    pred_id = int(probs.argmax().item())
    pred_label = ID2LABEL[pred_id]
    confidence = float(probs[pred_id].item())
    return pred_label, confidence, probs


if __name__ == "__main__":
    # 1) Train and save model weights
    train_and_save()

    # 2) Example inference (uncomment and set your image path):
    # model = load_trained_model("ham10000_resnet18_stream.pth")
    # label, conf, probs = predict_image(model, "your_image.jpg")
    # print("Predicted:", label, "confidence:", conf)


Device: cpu
Epoch 1/5 | loss=1.0486 | train_acc=0.6875 | val_acc=0.0008
Epoch 2/5 | loss=1.1366 | train_acc=0.6567 | val_acc=0.0008
Epoch 3/5 | loss=1.1711 | train_acc=0.6477 | val_acc=0.0008
Epoch 4/5 | loss=1.2549 | train_acc=0.6190 | val_acc=0.0008
Epoch 5/5 | loss=1.3149 | train_acc=0.5672 | val_acc=0.0008
Saved: ham10000_resnet18_stream.pth


In [4]:
import torch
from torch import nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LABELS = ["nv","mel","bkl","bcc","akiec","vasc","df"]
NUM_CLASSES = 7

# Same preprocessing as validation
val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def load_model(weights_path="ham10000_resnet18_stream.pth"):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    state = torch.load(weights_path, map_location=DEVICE)
    model.load_state_dict(state)
    model = model.to(DEVICE).eval()
    return model

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        # hooks
        target_layer.register_forward_hook(self._forward_hook)
        target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inp, out):
        self.activations = out.detach()

    def _backward_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, x, class_idx=None):
        self.model.zero_grad()
        logits = self.model(x)

        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())

        score = logits[:, class_idx].sum()
        score.backward(retain_graph=True)

        # activations: [B, C, H, W]
        # gradients:   [B, C, H, W]
        grads = self.gradients[0]       # [C,H,W]
        acts = self.activations[0]      # [C,H,W]

        # global average pool gradients -> weights per channel
        weights = grads.mean(dim=(1, 2))  # [C]

        # weighted sum of activations
        cam = (weights[:, None, None] * acts).sum(dim=0)  # [H,W]
        cam = torch.relu(cam)

        # normalize 0..1
        cam -= cam.min()
        cam /= (cam.max() + 1e-8)
        return cam.cpu().numpy(), class_idx, torch.softmax(logits, dim=1)[0].detach().cpu().numpy()

def show_gradcam(image_path, weights_path="ham10000_resnet18_stream.pth"):
    model = load_model(weights_path)

    # ResNet18 last conv layer is layer4[-1].conv2 (good Grad-CAM target)
    target_layer = model.layer4[-1].conv2
    gradcam = GradCAM(model, target_layer)

    # Load image for display
    pil_img = Image.open(image_path).convert("RGB")
    disp_img = pil_img.resize((224, 224))
    img_np = np.array(disp_img)

    # Model input
    x = val_tfms(pil_img).unsqueeze(0).to(DEVICE)

    cam, pred_idx, probs = gradcam(x)

    # Convert CAM to heatmap
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    # Overlay heatmap on image
    overlay = (0.55 * img_np + 0.45 * heatmap).astype(np.uint8)

    # Print prediction
    pred_label = LABELS[pred_idx]
    pred_conf = probs[pred_idx]

    print(f"Prediction: {pred_label}  (confidence: {pred_conf:.3f})")

    # Plot
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.title("Original"); plt.axis("off"); plt.imshow(img_np)
    plt.subplot(1,3,2); plt.title("Grad-CAM Heatmap"); plt.axis("off"); plt.imshow(heatmap)
    plt.subplot(1,3,3); plt.title("Overlay"); plt.axis("off"); plt.imshow(overlay)
    plt.show()

# ---- USE IT ----
# Put your image path here:
# show_gradcam("your_image.jpg")
